# Stopping and shine-through

Attenuate a beam along its path through the plasma: where its atoms are ionised (the birth profile), how much shines through onto the wall, and how both depend on the density and the beam energy.

A beam stream decays as `dI/ds = -n_e sigma_s I`, with the effective stopping cross section `sigma_s` of every ionising channel: ionisation by electrons and ions and charge exchange, including multi-step paths through excited states. `deposit` traces each marker on a straight line in steps `ds`, adds `n_e sigma_s ds` to the optical depth at each step inside the last closed surface, and tallies the atoms each step ionises on flux shells.

In [ ]:
import numpy as np
import plasma_plots
import xarray as xr

import freeofcharge as foc
from freeofcharge.profiles import IonSpecies

plasma_plots.set_backend("plotly")  # interactive figures, embedded in the docs

stopping = foc.SuzukiStopping()
energy_per_amu = np.geomspace(20e3, 2e6, 60)  # eV
sigma = {}
for ne in (1e19, 1e20):
    pure = np.asarray(stopping(energy_per_amu, ne, 1e4, [ne], [2], [1]))
    helium = np.asarray(
        stopping(energy_per_amu, ne, 1e4, [0.9 * ne, 0.05 * ne], [2, 4], [1, 2])
    )
    assert np.all(np.diff(pure[energy_per_amu > 100e3]) < 0)  # falls with energy
    assert np.all(helium > pure)  # impurities stop the beam faster
    sigma[f"D, n_e = {ne:.0e}"] = pure
    sigma[f"D + 5 % He, n_e = {ne:.0e}"] = helium

low, high = (
    np.asarray(stopping(500e3, ne, 1e4, [ne], [2], [1])) for ne in (1e19, 1e20)
)
assert high > low  # multi-step ionisation through excited states grows with density

sigma = xr.Dataset(
    {name: ("energy", values) for name, values in sigma.items()},
    coords={
        "energy": (
            "energy",
            energy_per_amu / 1e3,
            {"long_name": "beam energy", "units": "keV/amu"},
        )
    },
).to_array("plasma", name="sigma_s")
sigma.attrs["units"] = "m^2"
sigma.plasma.plot.cross_sections(markers=False, title="Suzuki beam stopping")

`SuzukiStopping` is the fit of Suzuki et al. (Plasma Phys. Control. Fusion 40, 2097, 1998) for H, D and T beams in hydrogen plasmas with He, B, C, N, O and Fe impurities, from 10 keV/amu to 10 MeV/amu, as implemented in ASCOT5; the tests reproduce ASCOT5's values. The cross section falls with the beam energy, so faster beams reach deeper, and rises with the density and with impurities. The paper fits 10 to 100 and 100 to 10000 keV/amu separately, and the two fits differ by a few per cent where they meet; `SuzukiStopping` blends them across 100/1.1 to 110 keV/amu so the cross section is continuous (`SuzukiStopping(blend=1.0)` gives the published fits with their step).

## A check against Beer-Lambert

With a constant cross section (`ConstantStopping`) in a plasma of constant density, the shine-through of a pencil beam is `exp(-n sigma L)`, with `L` the length of its path inside the last closed surface. The path length below is measured independently, on a much finer grid than the deposition's step:

In [ ]:
equilibrium = foc.Equilibrium.solovev(6.2, 1.8, 1.7, 5.3, wall_psi_n=1.3)
origin, direction = foc.tangential_injection(5.3, 12.0)
pencil = foc.NeutralBeam.gaussian(origin, direction, 1.0e6, 1.0e6, markers=1)

s = np.arange(0.0, 30.0, 1e-4)
points = origin[None, :] + s[:, None] * direction[None, :]
length = (
    np.asarray(
        equilibrium.inside(np.hypot(points[:, 0], points[:, 1]), points[:, 2])
    ).sum()
    * 1e-4
)
print(f"path length in the plasma: {length:.3f} m")

n = 5e19
flat = foc.Plasma(
    rho=np.array([0.0, 1.0]),
    ne=np.full(2, n),
    te=np.full(2, 1e4),
    ti=np.full(2, 1e4),
    ions={"D": IonSpecies(2, 1, np.full(2, n))},
)
depth, shine = [], []
for sigma in np.geomspace(1e-22, 1e-20, 9):
    result = foc.deposit(
        pencil,
        equilibrium,
        flat,
        foc.ConstantStopping(sigma),
        step=0.005,
        births_per_marker=0,
    )
    depth.append(n * sigma * length)
    shine.append(result.shine_through_fraction)
depth, shine = np.array(depth), np.array(shine)
np.testing.assert_allclose(shine, np.exp(-depth), rtol=1e-3)

measured = xr.DataArray(
    shine,
    dims="depth",
    coords={"depth": ("depth", depth, {"long_name": "optical depth n sigma L"})},
    name="shine-through fraction",
)
measured.plasma.plot.against_theory(
    lambda tau: np.exp(-tau), logy=True, title="exp(-n sigma L)"
)

## Density and energy

With Suzuki's cross sections and parabolic profiles, the shine-through falls steeply with the density: the reason a beam may only be switched on above a minimum density. A faster beam shines through more but deposits deeper. The power balance closes for every run: what is not absorbed shines through.

In [ ]:
ions = {"D": (2, 1, 0.96), "He": (4, 2, 0.02)}
rho_edges = np.linspace(0.0, 1.0, 21)
densities = np.array([1e19, 2e19, 3e19, 5e19, 7e19, 1e20])
energies = (0.5e6, 1.0e6)
shine = {}
for energy in energies:
    beam = foc.NeutralBeam.gaussian(
        origin,
        direction,
        energy,
        16.5e6,
        width=0.3,
        divergence=0.005,
        markers=400,
        seed=1,
    )
    shine[energy] = []
    for ne0 in densities:
        plasma = foc.Plasma.parabolic(ne0, 1e4, ions=ions)
        result = foc.deposit(
            beam,
            equilibrium,
            plasma,
            step=0.03,
            births_per_marker=0,
            rho_edges=rho_edges,
        )
        balance = (
            result.absorbed_power + result.shine_through_power - result.injected_power
        )
        assert abs(balance) < 1e-9 * result.injected_power
        assert abs(result.power.sum() / result.absorbed_power - 1) < 1e-9
        shine[energy].append(result.shine_through_fraction)
    shine[energy] = np.array(shine[energy])
    assert np.all(np.diff(shine[energy]) < 0)  # denser: less shine-through
assert np.all(shine[1.0e6] > shine[0.5e6])  # faster: more shine-through

shine_through = xr.DataArray(
    100 * np.array([shine[energy] for energy in energies]),
    dims=("energy", "ne0"),
    coords={
        "energy": (
            "energy",
            np.array(energies) / 1e6,
            {"long_name": "beam energy", "units": "MeV"},
        ),
        "ne0": (
            "ne0",
            densities,
            {"long_name": "central electron density", "units": "m^-3"},
        ),
    },
    name="shine-through",
    attrs={"units": "%"},
)
shine_through.plasma.plot.profiles(
    x="ne0", over="energy", logy=True, cmap="tab10", title=""
)

## Birth profile and wall hits

The birth profile is the power deposited per shell, divided by the shell volumes for a density. The fast ions are also sampled (`births_per_marker`), each with its birth point, velocity and pitch `v_par / v`; they feed the slowing down. With a wall, `wall_hits` holds where the shine-through leaves the vessel.

In [ ]:
surfaces = equilibrium.flux_surfaces(rho_edges)
beam = foc.NeutralBeam.gaussian(
    origin, direction, 1.0e6, 16.5e6, width=0.3, divergence=0.005, markers=400, seed=1
)
results = {}
for ne0 in (2e19, 1e20):
    plasma = foc.Plasma.parabolic(ne0, 1e4, ions=ions)
    results[ne0] = foc.deposit(
        beam,
        equilibrium,
        plasma,
        step=0.03,
        births_per_marker=2,
        rho_edges=rho_edges,
        seed=2,
    )
    print(
        f"n_e0 = {ne0:.0e}: shine-through {100 * results[ne0].shine_through_fraction:.1f} %"
    )
peaks = [np.average(surfaces.rho, weights=result.power) for result in results.values()]
assert peaks[1] > peaks[0]  # denser: deposited further out

power = xr.concat(
    [results[ne0].to_dataset(surfaces).power_density for ne0 in results],
    dim=xr.DataArray(
        list(results),
        dims="ne0",
        name="ne0",
        attrs={"long_name": "n_e0", "units": "m^-3"},
    ),
)
power.plasma.plot.profiles(x="rho", over="ne0", cmap="tab10").show()

phi = np.linspace(0, 2 * np.pi, 400)
with plasma_plots.figure(figsize=(6, 5)) as fig:
    ax = fig[0]
    for ne0, result in results.items():
        births = np.asarray(result.births.positions)
        ax.plot(
            births[:, 0],
            births[:, 1],
            ".",
            markersize=2,
            label=f"births, n_e0 = {ne0:.0e}",
        )
    hits = np.asarray(results[2e19].wall_hits.positions)
    ax.plot(
        hits[:, 0],
        hits[:, 1],
        "x",
        color="k",
        markersize=4,
        label="shine-through on the wall",
    )
    for radius in (
        equilibrium.boundary.vertices[:, 0].min(),
        equilibrium.boundary.vertices[:, 0].max(),
    ):
        ax.plot(radius * np.cos(phi), radius * np.sin(phi), color="0.6", linewidth=0.8)
    ax.set_aspect("equal")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.legend(fontsize=7, loc="lower left")

Charge exchange is part of `sigma_s`: an ion that picks up an electron and leaves as a fast neutral counts as stopped where it was ionised, not followed further. Each marker is a straight line, so the beam is not deflected and no orbit is followed before ionisation.